<div class="alert alert-block alert-info">

# Part 3A: Machine Learning: Loading data for Training and Test Sets and Creating a Naïve Bayes Model

## Preparation for model building

We are building multiple supervised learning models to predict the biological activity of small molecules. In this context, the input data are represented by MACCS Keys. The output data is the compound’s activity, encoded as `1` for active and `0` for inactive.

The model we are building learns a mathematical relationship between the input features and the output activity. This can be represented by the equation: 

<center>y=<i>f</i>(X)</center>

where:
* y is the predicted activity (the output),
* X is a vector of descriptors (in our case, the MACCS keys),
* *f* is the model that maps features to a predicted outcome.

We use uppercase X to indicate that the input data is multiple descriptors

### Loading the data into X and y.

We need to load the saved activity/MACCS Keys into a dataframe.

In [ ]:
import pandas as pd
df_data = pd.read_csv("AID743139_activity_MACCS.csv")

In [ ]:
# we now have a dataframe with CIDS, activities and maccs keys
df_data.head(3)

We will put the MACCS Keys into a variable called X_MACCS.<br>
We will put the activity values into a variable called y.

In [ ]:
X_MACCS = df_data.iloc[:,2:] # this is dropping cid and activity and creating a new variable for maccs data
y = df_data['activity'].values

In [ ]:
X_MACCS.head(3)

In [ ]:
print(len(y))    # Number of all compounds
y.sum()          # Number of actives

### Remove zero-variance features

Some features in X are not helpful in distinguishing actives from inactives, because they are set ON for all compounds or OFF for all compounds.  Such features need to be removed because they would consume more computational resources without improving the model.

We will use the `VarianceThreshold` method of sklearn to identify which features have a variance of zero or very low. Variance in data represents how spread out the values of a feature are. The `threshold` parameter is set to 0.0 by default, meaning only features with zero variance (constant values across all samples, 100% identical values) are removed. 
<div class="alert alert-block alert-info">
<details>
<summary>What if a feature has ≥99% identical values?</summary>
Let’s say a feature is <code>1</code>  in 99.5% of rows and <code>0</code>  in the remaining 0.5%. It does not have zero variance, but the variance is very low.

If you want to remove such near-constant features, you need to set <code>threshold</code> accordingly. In this case the variance is calculated as:
<center>Var(X)=<i>p</i>(1−<i>p</i>)=0.995×(1−0.995)=0.004975<br>
where <i>p</i> is probability of the feature being 1</center>


So to remove this feature, your threshold must be greater than 0.004975, for example:
<code>VarianceThreshold(threshold=0.005)</code>

It might be interesting to see how our models change, or time calculating the model changes if we do some prefiltering by adjusting the threshold.


In [ ]:
from sklearn.feature_selection import VarianceThreshold

In [ ]:
X_MACCS.shape  #- Before removal

In [ ]:
# Code to filter out features with low variance and save to a new dataframe
# Allows us to play with threshold value to remove features with low variance

sel = VarianceThreshold(threshold=0.00) # while the default is 0.0, we can set this to a different value.
X_MACCS_filtered=sel.fit_transform(X_MACCS) # filters and removes columns with variance below the threshold in one step

# the get_support() method returns a boolean mask indicating which features were kept (True) or removed (False)
mask = sel.get_support() 

#use the mask to filter the columns in the original DataFrame
kept_features = X_MACCS.columns[mask]
removed_features = X_MACCS.columns[~mask] # ~ This will give us the features that were removed

print("Features removed:", removed_features)
X_MACCS_filtered.shape  #- After removal

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.

Why is maccs000 removed regardless of threshold level?

Make note of the MACCS Keys removed as we will need these later in the activity.


### Train-Test-Split (a 9:1 ratio)

Now that we’ve prepared the dataset, the next step is to divide it into two parts: one for training the model and one for testing it. This is important because we want to evaluate how well the model performs on unseen data, and not just the data it was trained on.

This is typically done by splitting the dataset into two subsets using a specified ratio. Common splits include 80:20 or 70:30, where the larger portion is used for training and the smaller for testing. When the dataset is small or the model requires more examples to learn effectively, a 90:10 split can be helpful.

In the next code section, we will split the data so that 90% goes into the training set and 10% into the test set. The training set is used to build the model, while the test set is used to evaluate how well the model generalizes to new data. 

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.

1) How many molecules are in the dataset?
2) How many molecules should be in the training set?
3) How many molecules should be in the test set?


In the following code, `X_train` and `X_test` contain the molecular fingerprint data used by the model. The X values are the input features, meaning the structural information for each molecule. `X_train` contains the compounds used to train the model, while `X_test` contains compounds held back to evaluate how well the model performs on data it has not seen before. The variables `y_train` and `y_test` contain the known activity labels for those same compounds. `y_train` provides the activity labels used during training, and `y_test` provides the correct answers used to evaluate the model’s predictions. Since `test_size=0.1`, 10% of the dataset is placed in the test set and the remaining 90% is used for training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(X_MACCS_filtered, y, shuffle=True, random_state=3100, stratify=y, test_size=0.1) # test_size = 0.1 is 10% of the data set

print("Training set shape:", X_train.shape, y_train.shape)
print("where there are", X_train.shape[0], "samples, and", X_train.shape[1], "features")
print("and", y_train.shape[0], "activities associated with the training set.")
print()
print("Test set shape:", X_test.shape, y_test.shape)
print("where there are", X_test.shape[0], "samples, and", X_test.shape[1], "features")
print("and", y_test.shape[0], "activities associated with the test set.")
print()
print("Number of active compounds in training set:", y_train.sum())
print("Number of active compounds in test set:", y_test.sum())

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.

1) How many molecules are in the full dataset?
2) How many molecules **are** in the **training set**?
3) How many molecules **are** in the **test set**?
4) Were your predictions correct? If not, explain.

### Balancing the training set

Before training a classification model, it is important to examine the number of active versus inactive compounds in the training set. This helps assess whether the dataset is:

* **balanced**, where both classes are represented in roughly equal proportions or,
* **unbalanced**, where one class significantly outnumbers the other. 

A balanced dataset is ideal for training because the model can learn to distinguish between both classes (active vs inactive) effectively. In contrast, an unbalanced dataset may cause the model to become biased toward the majority class, leading to poor performance in predicting the minority class. 

By checking the class distribution early, we can decide whether additional steps, such as resampling or using class weights, are needed to improve model fairness and accuracy.

Check the number of actives and inactive compounds in the training set.

In [ ]:
print("# inactives in training set: ", len(y_train) - y_train.sum())
print("# actives in training set:   ", y_train.sum())
ratio = (len(y_train) - y_train.sum())/y_train.sum()
print("the ratio of inactive to active in training set=", ratio)

<div class="alert alert-block alert-info">


<details>
<summary>How to interpret class balance ratios.</summary>

| **Ratio**|  **How to Interpret the Value** |
|----------:|---------------------------------|
|1|Balanced: Roughly equal number of active and inactive. Ideal for training|
|2|Mild imbalance: 1 active for every 2 inactives. Still manageable, but performance of minority class should be monitored.|
|>5| Severe imbalance. Model may predict majority class most of the time and ignore the minority class.|

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.

1) Is your training set balanced?
2) Based on the ratio of inactive to active in the training set, would you expect a model built with this data to do a better job at predicting active or inactive compounds?


#### Downsampling
When majority class in a dataset is unbalanced, downsampling is used select a subset of the majority class to minority class. This helps the model learn to distinguish the classes more fairly. While it reduces data quality, it can improve model performance on the minority class, and prevent biases toward the majority. Downsampling is not without risk, however, as we may be discarding potentially useful information from the majority class.

In the code cell below, we will randomly select from the inactive molecules a number of compounds that is equal to active molecules.

In this code, `X_train_downsampled` and `y_train_downsampled` are new versions of the training data created to balance the number of active and inactive compounds. The original training set contains many more inactive compounds than active compounds, so the code randomly selects a smaller subset of inactive compounds equal to the number of active compounds. `X_train_downsampled` contains the molecular fingerprint features for this balanced set of compounds, while `y_train_downsampled` contains the corresponding activity labels. The original `X_train` and `y_train` are left unchanged, so we can compare the original imbalanced training set to the downsampled balanced training set. Only the training data are downsampled; the test set remains unchanged so that model performance can still be evaluated on the original distribution of compounds.

In [ ]:
# load the numpy libraray
import numpy as np

# Indicies of each class' observations
idx_inactives = np.where( y_train == 0 )[0]
idx_actives   = np.where( y_train == 1 )[0]

# Number of observations in each class
num_inactives = len(idx_inactives)
num_actives   = len(idx_actives)

# Randomly sample from inactives without replacement
# setting size to the number of actives ensures we downsample inactives to match the number of actives
np.random.seed(0)  #sets the random seed for reproducibility. You might want to try a value of 2026 after you get your first confusion matix.
idx_inactives_downsampled = np.random.choice(idx_inactives, size=num_actives, replace=False)

# Join together downsampled inactives with actives
# vstack and hstack are used to combine arrays vertically and horizontally, respectively
# this ensures that the rows from downsampled inactives and actives are combined correctly
# we use vstack for a 2D array (X_train) and hstack for a 1D array (y_train)
X_train_downsampled = np.vstack((X_train[idx_inactives_downsampled], X_train[idx_actives]))
y_train_downsampled = np.hstack((y_train[idx_inactives_downsampled], y_train[idx_actives]))

#X_train = np.vstack((X_train[idx_inactives_downsampled], X_train[idx_actives]))
#y_train = np.hstack((y_train[idx_inactives_downsampled], y_train[idx_actives]))


#confirm the downsampling worked
print("# inactives orig: ", len(y_train) - y_train.sum())
print("# inactives downsampled: ", len(y_train_downsampled) - y_train_downsampled.sum())

print("# actives orig  : ", y_train.sum())
print("# actives downsampled  : ", y_train_downsampled.sum())
ratio = (len(y_train) - y_train.sum())/y_train.sum()
print("the ratio of active to inactive original=", ratio)
ratio_downsampled = (len(y_train_downsampled) - y_train_downsampled.sum())/y_train_downsampled.sum()
print("the ratio of active to inactive downsampled=", ratio_downsampled)
print()
# check to see the number of samples and features in the training set
print("Training set shape:", X_train.shape, y_train.shape)
print("where there are", X_train.shape[0], "samples, and", X_train.shape[1], "features")
print("and", y_train.shape[0], "activities associated with the training set.")
print()
print("Downsampled Training set shape:", X_train_downsampled.shape, y_train_downsampled.shape)
print("where there are", X_train_downsampled.shape[0], "samples, and", X_train_downsampled.shape[1], "features")
print("and", y_train_downsampled.shape[0], "activities associated with the training set.")


We now have set up variables to train our model on either the original data set or in the downsampled data set.

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.

1) Is your downsampled training set balanced? How many actives and inactives are found in the downsampled training set?
2) Based on the ratio of inactive to active in the downsampled training set, would you expect a model built with this data to do a better job at predicting active or inactive compounds? Explain.

## Building a model using the training set

Now we are ready to build predictive models using machine learning algorithms available in the scikit-learn library (https://scikit-learn.org/).  This notebook will use Naïve Bayes and decision tree, because they are relatively fast and simple.

In [ ]:
from sklearn.naive_bayes import BernoulliNB        #-- Naïve Bayes


In [ ]:
from sklearn.metrics import classification_report #provides detailed report that includes precision and sensitivity
from sklearn.metrics import confusion_matrix      # gives a 2x2 matrix for true netatives, false positives, false negatives and true positives
from sklearn.metrics import accuracy_score        # computes accuracy = number of correct preictions/total number of preditions
from sklearn.metrics import roc_auc_score         # Computs Area under the ROC curve, evaluates trade-off of true positive rate and false positive rate

### Creating a Naïve Bayes Model

Bernoulli Naïve Bayes is a probabilistic classification algorithm based on Bayes’ Theorem, well-suited for binary feature vectors such as molecular fingerprints. In cheminformatics, it is often applied to classify molecules (e.g., active vs. inactive) based on structural features represented as binary indicators — such as the presence or absence of specific substructures captured by MACCS keys or other molecular fingerprints. The algorithm assumes that features are conditionally independent given the class label and models each feature using a Bernoulli (0 or 1) distribution. During training, it learns the likelihood of each structural feature being present within each class. Despite its simplicity and the strong independence assumption, Bernoulli Naïve Bayes is efficient and effective for high-dimensional molecular data, making it a useful baseline model for structure-activity classification tasks.

In [ ]:
# set up the NB classification model. Bernoulli is specific for binary features (0,1) 
clf_NB = BernoulliNB()            

The next line of code trains, or “fits,” the Bernoulli Naïve Bayes classifier using the selected training data. In this notebook, you may choose to train the model using either the original training set or the downsampled training set by changing which line is commented out.

The input feature matrix, such as `X_train` or `X_train_downsampled`, contains the molecular fingerprint data used for training. Each row represents one compound, and each column represents one fingerprint feature, typically encoded as a 0 or 1. The corresponding target labels, such as `y_train` or `y_train_downsampled`, identify whether each compound is classified as inactive or active.

During training, the Bernoulli Naïve Bayes classifier estimates how often each fingerprint feature is present in active compounds and how often it is present in inactive compounds. It also calculates the prior probability of each class based on the class labels in the selected training set. These probabilities are then used later to predict the activity class of compounds in the test set.

In [ ]:
# Train the model by fitting it to the data.

# Option 1: Create a model based on the original dataset
#clf_NB.fit( X_train ,y_train )   

# Option 2:Create a model based on the downsampled dataset
clf_NB.fit( X_train_downsampled ,y_train_downsampled ) 

In [ ]:
y_train_downsampled.shape

The next code cell prepares the labels needed to evaluate how well the trained model performs on the training data. The variable `y_true` contains the known activity labels for the selected training set. The variable `y_pred` contains the activity labels predicted by the trained model for that same set of compounds.

In this notebook, you can evaluate the model using either the original training set or the downsampled training set. The important point is that `y_true` and `y_pred` must come from the same version of the training data. If you use `y_train`, then the predictions should be made from `X_train`. If you use `y_train_downsampled`, then the predictions should be made from `X_train_downsampled`.

In [ ]:
# Apply the model to predict the activity of compounds in the training set.

# Option 1: Evaluate performance on the original training set
# y_true, y_pred = y_train, clf_NB.predict(X_train)

# Option 2: Evaluate performance on the downsampled training set
y_true, y_pred = y_train_downsampled, clf_NB.predict(X_train_downsampled)

Next, we will use a confusion matrix to evaluate how well our model predicts active and inactive compounds. A confusion matrix compares the actual labels (from the test set) with the predicted labels (from the model) and summarizes the results in a structured way.

We use `confusion_matrix()` from `sklearn.metrics` to generate this matrix. For binary classification, the output is a 2×2 NumPy array, commonly referred to as CMat, with the following layout:
<div style="font-family: Arial, sans-serif; margin-top: 20px;">
<center>
  <h4>Confusion Matrix (Binary Classification)</h4>
  <table border="1" cellspacing="0" cellpadding="10" style="border-collapse: collapse; text-align: center;">
    <tr>
      <th rowspan="2">Actual</th>
      <th colspan="2">Predicted</th>
    </tr>
    <tr>
      <th>0</th>
      <th>1</th>
    </tr>
    <tr>
      <th>0</th>
      <td>TN<br><small>True Negative</small></td>
      <td>FP<br><small>False Positive</small></td>
    </tr>
    <tr>
      <th>1</th>
      <td>FN<br><small>False Negative</small></td>
      <td>TP<br><small>True Positive</small></td>
    </tr>
  </table>
</center>
</div>


This matrix gives us insight into the types of errors the model makes and is the foundation for calculating metrics like accuracy, precision, recall, and F1-score.


In [ ]:
 #-- generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

<div class="alert alert-block alert-warning">
<strong>Interpreting the confusion matrix</strong> Write your answers in the following raw cell.

1) How does the sum of the predictions relate to number of molecules the training set? 

2) How many actives are in the training set? How many would a perfect model predict? Is this predicting more or less?

3) How many inactives are in the training set? How many would a perfect model predict? Is this predicting more or less?

4) Is this model perfect? Is it *"good enough"*?

To answer the question “Is the model good enough?”, we rely on key metrics calculated from the confusion matrix:

- Accuracy: The proportion of all predictions that were correct = (TP+TN)/(TP+TN+FP+FN)
- Precision: Of the compounds the model predicted as active, how many were actually active? = TP / (TP + FP)
- Sensitivity: Of the truly active compounds, how many did the model correctly identify? = TP / (TP + FN)
- Specificity: measures how well model identifies actual negatives = TN / (TN + FP )
- Balanced accuracy: averages sensitivity and specificity = (sens + spec) / 2 
- F1-score: Harmonic mean of precision and recall =2 × (Precision × Specificity) / (Precision + Specificity)
- AUC-ROC (Area Under the Receiver Operating Characteristic Curve)- Measures the model's ability to distinguish between classes

These metrics help us assess not only overall accuracy but also the types of errors the model makes. For example:

* Is the model missing many actives? (high FN → low sensitivity)
* Is it misclassifying inactives as actives? (high FP → low precision)

Understanding these trade-offs is essential for deciding whether the model is acceptable, especially in fields like drug discovery, where missing an active might be more costly than flagging a few false positives.

In [ ]:
# Choose which training set to evaluate
# Option 1: Original training set
#X_eval = X_train
#y_true = y_train
#set_name = "Original training set"

# Option 2: Downsampled training set
X_eval = X_train_downsampled
y_true = y_train_downsampled
set_name = "Downsampled training set"

# Apply the trained model to the selected training set
y_pred = clf_NB.predict(X_eval)
# Get predicted probabilities for the positive class, label 1
y_score = clf_NB.predict_proba(X_eval)[:, 1]

# Calculate confusion matrix values
TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()

# Calculate performance metrics
acc  = accuracy_score(y_true, y_pred)            # (TP + TN) / (TP + TN + FP + FN)
prec = TP / (TP + FP)                            # Precision: how many predicted positives were actually positive
sens = TP / (TP + FN)                            # Sensitivity: how many actual positives were identified
spec = TN / (TN + FP)                            # Specificity: how many actual negatives were identified
bacc = (sens + spec) / 2                         # Balanced accuracy: average of sensitivity and specificity
f1_score = 2 * (prec * sens) / (prec + sens)     # Harmonic mean of precision and sensitivity

auc = roc_auc_score(y_true, y_score)             # Measures how well the model ranks positives above negatives

# Print results
print(f"{set_name} performance metrics:")
print(f"Accuracy          = {acc:.4f}")
print(f"Precision         = {prec:.4f}")
print(f"Sensitivity       = {sens:.4f}")
print(f"Specificity       = {spec:.4f}")
print(f"Balanced Accuracy = {bacc:.4f}")
print(f"F1 Score          = {f1_score:.4f}")
print(f"AUC-ROC           = {auc:.4f}")

<div class="alert alert-block alert-info">
<details>
<summary>Classification Metrics Interpretation Guide</summary>


| **Metric**| **What It Measures** | **How to Interpret the Value** |
|:--------|:--------|:--------|
|  **Accuracy**   | Overall % of correct predictions | ✅ ≥ 0.80 = High accuracy<br>⚠️ 0.60–0.79 = Moderate *Can be misleading with class imbalance* <br>🔴 < 0.60 = Poor accuracy|
|  **Balanced Accuracy**   |  Avg. of sensitivity (recall) and specificity  |  ✅ ≥ 0.80 = Strong<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Poor|
|  **Sensitivity** (Recall)  |  % of actual actives correctly predicted (TPR / Recall)   | ✅ ≥ 0.80 = Few missed positives<br>⚠️ 0.60–0.80 = Moderate<br>🔴 < 0.60 = Many positives missed  |
|  **Specificity**   |  % of actual inactives correctly predicted (TNR)  |  ✅ ≥ 0.80 = Few false positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives  |
|  **Precision**   |  % of predicted positives that are truly positive  |  ✅ ≥ 0.80 = Few false positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives  |
|**F1-score**| harmonic Mean of precision and recall|✅ ≥ 0.75 = Few false Positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives
|  **AUC-ROC**|  Ability to rank actives above inactives across thresholds |  ✅ ≥ 0.80 = Strong discrimination<br>⚠️ 0.70–0.79 = Acceptable<br>🔴 < 0.70 = Weak model<br>🚫 ~0.50 = Random guessing |

**Quick Rules of Thumb:**
- If accuracy is high but balanced accuracy is low → suspect class imbalance.
- Use balanced accuracy when your dataset has a lot more inactives than actives.
- Sensitivity is important when missing an active could be costly (e.g., in drug screening).
- Specificity is important when false positives are costly (e.g., expensive follow-up experiments).
- AUC-ROC gives a broader view of model quality — useful even when you're not picking a classification threshold yet.

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the confusion matrix</strong> Write your answers in the following  raw cell.

1) Using the Classification Metrics Interpretation Guide, how well did the model predict those molecules in the training set?
2) Why is the value for `Balanced Accuracy` identical to `Accuracy` in the downsampled set?

While these values represent ability to predict the test set, the real performance of the model should be evaluated with the test set data, which are not used for model training. The test set should not be downsampled. It is held back as an independent evaluation set so you can see how the trained model performs on compounds it did not see during training.

In [ ]:

# Apply the trained model to predict the activity of compounds in the test set

X_eval = X_test
y_true = y_test
set_name = "Test set"

y_pred = clf_NB.predict(X_eval)


In [ ]:
CMat = confusion_matrix( y_true, y_pred )    #-- generate confusion matrix
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]

# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
y_score = clf_NB.predict_proba(X_eval)[:, 1]

TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()

acc  = accuracy_score(y_true, y_pred)
prec = TP / (TP + FP)
sens = TP / (TP + FN)
spec = TN / (TN + FP)
bacc = (sens + spec) / 2
f1_score = 2 * (prec * sens) / (prec + sens)

auc = roc_auc_score(y_true, y_score)

print(f"{set_name} performance metrics:")
print(f"Accuracy          = {acc:.4f}")
print(f"Precision         = {prec:.4f}")
print(f"Sensitivity       = {sens:.4f}")
print(f"Specificity       = {spec:.4f}")
print(f"Balanced Accuracy = {bacc:.4f}")
print(f"F1 Score          = {f1_score:.4f}")
print(f"AUC-ROC           = {auc:.4f}")

print()
print("Check the number of actives and inactive compounds in the test set.")
print("# inactives : ", len(y_test) - y_test.sum())
print("# actives   : ", y_test.sum())
ratio = (len(y_test) - y_test.sum())/y_test.sum()
print("the ratio of active to inactive =", ratio)
print()

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the confusion matrix for the Test set</strong>

1) Using the Classification Metrics Interpretation Guide, how well did the model predict those molecules in the test set?

2) Why is the balanced accuracy value different from the accuracy in the test set example?

3) How does our precision compare from the training to the test set?

4) How do the F1 values compare from training to test set?

Some additional performance information may be obtained using scikit-learn's **classification_report()**.  It gives a summary of some of the items we have calculated so far (precision, recall/sensitivity, F1 and number of instances in each class called support). This can be very helpful when looking at imbalanced sets, like our test set, because it includes data for each class as well as weighted averages to take into account the the proportion of classes.  This allows for a more nuanced evaluation of the model across classes.

In [ ]:
print( classification_report(y_true, y_pred))

<div class="alert alert-block alert-info">


<details>
<summary>Interpreting additional metric information provided in the classification_report</summary>

| Metric | What It Measures | Interpretation | 
|--------------|---------------------------------------------------------------|----------------------------------------------------------------------------| 
| Support | Number of actual instances of the class in the dataset | Not a metric. It just tells you how many samples belong to each class. |

#### 

| Metric | What It Means | When to Use / Why It Matters | 
|----------------|--------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------| 
| **Macro Avg** | Average of each metric (precision, recall, F1) calculated **per class**, without considering class size. | Treats all classes equally. Highlights poor performance on minority classes. Good for imbalanced datasets when fairness across classes matters. | 
| **Weighted Avg**| Average of each metric weighted by the number of instances (**support**) in each class. | Reflects overall performance, but may hide poor performance on small classes. Good when class proportions should affect the overall score. |

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the classification report for the Test set</strong> Write you answers in the following raw cell.

1) How well does the model predict inactives in the test set?
2) How well does the model predict actives in the test set?


**Let's predict if a molecule is active!**

We can use the model we created to determine if a molecule is active or inactive the enzyme. The following lines of code define SMILES to test. Some of the following are in the training set and active, some in the training set and inactive, and some are outside the training set. A returned value of `1` = active, and `0` = inactive.

*Note: You should rerun the following two code cells multiple times, uncommenting each successive line in the next cell to change the SMILES string input.*

In [ ]:
from rdkit import Chem

#Define new SMILES string and view
new_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O" # CID 2242 aspirin should be INactive
#new_smiles = "C1=CC=C(C=C1)C(C2=CC=CC=C2)(C3=CC=CC=C3Cl)N4C=CN=C4" # CID = 2812 should be active
#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "CCCCCC1=CC(=C2C=CC(OC2=C1)(C)CCC=C(C)C)O" #CID30219 not in datbase (CBC)
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active
#new_smiles = "CCCCCC1=CC(=C2[C@@H]3C=C(CC[C@H]3C(OC2=C1)(C)C)C)O" #CID16078 in database and should be active (THC)
#new_smiles = "COC1=CC(=CC(=C1OC)OC)CCN" #CID4076 not in database (mescaline)
#new_smiles = "CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C" # vitamin A not in database
mol = Chem.MolFromSmiles(new_smiles)
mol

Note: The following code cell will return a warning that indicates the model was trained with named columns, but the new prediction input does not have those names. The warning is usually harmless.

In [ ]:
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import MACCSkeys
import numpy as np
fp = MACCSkeys.GenMACCSKeys(mol)
arr = np.zeros((1,), dtype=int) 
ConvertToNumpyArray(fp, arr)
arr_filtered = sel.transform(arr.reshape(1, -1))
prediction = clf_NB.predict(arr_filtered)
print(prediction)
if prediction ==0:
    print("Molecule is inactive.")
else:
    print("Molecule is active.")
    

<div class="alert alert-block alert-warning">
<strong>Testing the model</strong>

Test all the new_smiles in the model and add your own SMILES to test the model. Additional molecules you might want to test are androgens (like androstenedione) as they are key substrates for this enzyme. We also know that hydrogen bond donors and acceptors as well as aromatic features are important for interaction with the active site. Formestane and exmestane are known inhibitors. They may or may not be in the training set. It would be interesting to test them here.

How well is the Bernoulli Naïve Bayes model doing at your predictions? Do you trust it? If you are not satisfied, what would you do to improve it?

## Saving your model and your training data

Now that we have trained a model, we should save it for future use to avoid retraining. This can be achieved using either the `pickle` and `joblib` libraries. While `pickle` is the standard python library, `joblib` is often preferred for models with large NumPy arrays, such as our MACCS Keys, because it is more efficient for handling large data structures. 

One very important issue to consider is that while the `joblib` library is more efficient, it doesn't save all the information necessary to correctly prepare future molecules for prediction. Recall that we used a variance mask to remove MACCS Key positions that had zero variance in the training data. That same mask must be applied to a new molecule before making a prediction. That is why I asked you to note what MACCS Keys were removed with zero variance. We are going to need that information when making predictions with our saved models.

In [ ]:
# How to save the model using joblib

import joblib
model = clf_NB #our Naive Bayes model
filename = 'clf_NB_model_MACCS.joblib' # give our model a name for the file
joblib.dump(model, filename) # save the model to a file
print(f"Model saved to {filename}")
joblib.dump(sel, "maccs_variance_selector.joblib") #save which MACCS Keys were dropped


You can also save the split training and test sets as files, then load them into a new notebook. Since the `X_train`, `X_test`, `y_train`, and `y_test` and the corresponding downsampled variables `X_train_downsampled`, and `y_train_downsampled` are NumPy arrays, we can save them as them as .npy files.

We are also adding `MACCS163` to the file name to signify that these were all built with removal of the zero variance features.

In [ ]:
import numpy as np

np.save("X_train_MACCS163.npy", X_train)
np.save("X_train_downsampled_MACCS163.npy", X_train_downsampled)

np.save("y_train163.npy", y_train)
np.save("y_train_downsampled_MACCS163.npy", y_train_downsampled)

np.save("X_test_MACCS163.npy", X_test)
np.save("y_test163.npy", y_test)



### Realoading your training data

The following code section shows you how to reload your training data and your joblib model.

In [ ]:
# Python code to reload all of your training data

import numpy as np

X_train = np.load("X_train_MACCS163.npy")
X_train_downsampled = np.load("X_train_downsampled_MACCS163.npy")

y_train = np.load("y_train163.npy")
y_train_downsampled = np.load("y_train_downsampled_MACCS163.npy")

X_test = np.load("X_test_MACCS163.npy")
y_test = np.load("y_test163.npy")


In [ ]:
print(len(X_train[1]))
print(len(y_train_downsampled))
y_test.shape

### Realoading your model and testing compounds

The following code sections show you how to reload your joblib model. 

In [ ]:
# How to load the model using joblib
# import necessary libraries

import joblib, numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import ConvertToNumpyArray
from sklearn.feature_selection import VarianceThreshold


In [ ]:
# 1) Load your saved classifier (This was a Naive Bayes Model)
clf = joblib.load("clf_NB_model.joblib")

# 2) Load your saved variance selector
sel = joblib.load("maccs_variance_selector.joblib")

In [ ]:
# 2) Load a smiles to predict and generate MACCS Keys
smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin
mol = Chem.MolFromSmiles(smiles)
fp = MACCSkeys.GenMACCSKeys(mol)
print(len(fp))

In [ ]:
# 3a) Use the saved variance selector to drop the four MACCS that had zero variance and predict activity
arr = np.zeros((1,), dtype=int) 
ConvertToNumpyArray(fp, arr)
arr_filtered = sel.transform(arr.reshape(1, -1))
#prediction = clf_NB.predict(arr_filtered)
prediction = clf.predict(arr_filtered)

print(prediction)
if prediction ==0:
    print("Molecule is inactive.")
else:
    print("Molecule is active.")
    

Alternatively, we can tell specifically wich MACCS Keys to drop

In [ ]:
#3b) Drop the MACCS Keys that were dropped in generation of the model. In our case those were 0, 1, 2, and 4
# create a set of the values in the fingerprint we want to drop
MACCS_to_drop = {0,1,2,4} # use a set for faster lookup

# Keep items if their index is NOT in the removal set
fpdrop =[item for i, item in enumerate(fp) if i not in MACCS_to_drop]

print(len(fpdrop))


In [ ]:
# 4) Predict after manually dropping the four MACCS Keys
X_query = np.array(fpdrop).reshape(1, -1) # The classifier needs a 2D shape, but we have 1D list. We reshape (1, n_bits)
prediction = clf.predict(X_query)

if prediction == 0:
    print("Molecule is inactive for human aromatase.")
else:
    print("Molecule is active for human aromatase. Active may be agonist or antagonist in this model.")

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following code cell.

1) For CID 65906, determine whether the compound is known to be active or inactive. Then compare this known outcome to the model’s predicted class and predicted probabilities. What do the probability values tell you about the reliability of the model?In your answer, identify whether the model was confidently correct, uncertain but correct, uncertain and incorrect, or confidently incorrect.
2) Let's reconsider your earlier answer about whether you trusted the model. Does the probability information make you trust the model more or less? Explain your reasoning using the predicted probabilities and the known active/inactive state of the compound.